In [65]:
import pandas as pd
import pandapower as pp

buses = pd.read_csv("buses.csv")
lines = pd.read_csv("lines.csv")
trafos = pd.read_csv("trafos.csv")
gens = pd.read_csv("generators.csv")
loads = pd.read_csv("loads.csv")

print(len(buses), len(lines), len(trafos), len(gens), len(loads))

net = pp.create_empty_network(f_hz=60, sn_mva=100)

# CSV bus number -> pandapower bus index
bus_idx = {}
for _, row in buses.iterrows():
    bus_idx[int(row["Bus Number"])] = pp.create_bus(
        net, vn_kv=row["Nominal Voltage"], name=str(int(row["Bus Number"]))
    )

print(len(net.bus), "buses created")
print(sorted(buses["Nominal Voltage"].unique()))

500 466 131 90 206
500 buses created
[13.8, 138.0, 345.0]


In [66]:
# --- loads ---
for _, r in loads.iterrows():
    pp.create_load(net, bus=bus_idx[int(r["Bus Number"])],
                   p_mw=r["P Constant MW"], q_mvar=r["Q Constant MVAR"])
print("load P (MW):", round(net.load.p_mw.sum(), 1), "| csv:", round(loads["P Constant MW"].sum(), 1))

# --- generators: online only, bus 17 is the slack ---
SLACK = 17
for _, r in gens.iterrows():
    if int(r["Status"]) != 1:
        continue
    b = bus_idx[int(r["Bus Number"])]
    if int(r["Bus Number"]) == SLACK:
        pp.create_ext_grid(net, bus=b, vm_pu=r["Voltage Regulation Target PU"])
    else:
        pp.create_gen(net, bus=b, p_mw=r["Pgen"],
                      vm_pu=r["Voltage Regulation Target PU"])
print("gens:", len(net.gen), "+ 1 slack | other gen P (MW):", round(net.gen.p_mw.sum(), 1))

load P (MW): 7750.7 | csv: 7750.7
gens: 55 + 1 slack | other gen P (MW): 6954.5


In [67]:
import math

normal_mva, emerg_mva = [], []
for _, r in lines.iterrows():
    kv = r["Voltage"]
    zbase = kv**2 / 100.0                      # ohms, 100 MVA base
    r_ohm = r["R (pu @ 100MVA)"] * zbase
    x_ohm = r["X (pu @ 100MVA)"] * zbase
    b_siemens = r["B (pu @ 100MVA)"] / zbase
    c_nf = b_siemens / (2 * math.pi * 60) * 1e9
    max_i_ka = r["Normal Rating"] / (math.sqrt(3) * kv)

    pp.create_line_from_parameters(
        net,
        from_bus=bus_idx[int(r["From Bus Number"])],
        to_bus=bus_idx[int(r["To Bus Number"])],
        length_km=1.0,
        r_ohm_per_km=r_ohm, x_ohm_per_km=x_ohm, c_nf_per_km=c_nf,
        max_i_ka=max_i_ka,
        in_service=bool(r["Line Status"] == 1),
        name=f'{int(r["From Bus Number"])}-{int(r["To Bus Number"])}-{int(r["Circuit ID"])}',
    )
    normal_mva.append(r["Normal Rating"])
    emerg_mva.append(r["Emergency Rating"])

net.line["normal_mva"] = normal_mva
net.line["emerg_mva"] = emerg_mva
print(len(net.line), "lines created")
print(net.line[["name", "r_ohm_per_km", "x_ohm_per_km", "c_nf_per_km", "max_i_ka"]].head(3))

466 lines created
      name  r_ohm_per_km  x_ohm_per_km  c_nf_per_km  max_i_ka
0    2-1-1      0.101122      0.597921     0.000000  0.320053
1  3-479-1      1.382299     13.273573   313.393054  1.339997
2    4-3-1      0.131796      0.670035     0.000000  0.941583


In [68]:
trafo_normal, trafo_emerg = [], []
for _, r in trafos.iterrows():
    r_pu = r["R (pu @ 100MVA)"]
    x_pu = r["X (pu @ 100MVA)"]
    z_pu = math.sqrt(r_pu**2 + x_pu**2)

    pp.create_transformer_from_parameters(
        net,
        hv_bus=bus_idx[int(r["Bus Number Winding 2"])],   # higher-voltage side
        lv_bus=bus_idx[int(r["Bus Number Winding 1"])],   # lower-voltage side
        sn_mva=100.0,                                      # use the 100 MVA base
        vn_hv_kv=r["Winding 2 Voltage"],
        vn_lv_kv=r["Winding 1 Voltage"],
        vk_percent=z_pu * 100.0,
        vkr_percent=r_pu * 100.0,
        pfe_kw=0.0, i0_percent=0.0,
        in_service=bool(r["Status"] == 1),
        name=f'{int(r["Bus Number Winding 1"])}-{int(r["Bus Number Winding 2"])}-{int(r["Circuit ID"])}',
    )
    trafo_normal.append(r["Normal Rating"])
    trafo_emerg.append(r["Emergency Rating"])

net.trafo["normal_mva"] = trafo_normal
net.trafo["emerg_mva"] = trafo_emerg
print(len(net.trafo), "trafos created")
print(net.trafo[["name", "vk_percent", "vkr_percent", "normal_mva"]].head(3))

131 trafos created
    name  vk_percent  vkr_percent  normal_mva
0  8-7-1    1.290489     0.035506       567.0
1  9-7-2    1.362053     0.012000       680.0
2  9-7-1    1.362053     0.012000       680.0


In [69]:
pp.runpp(net, numba=False, init="dc")
print("converged:", net.converged)
print("slack P (MW):", round(net.res_ext_grid.p_mw.sum(), 1))
print("vm range:", round(net.res_bus.vm_pu.min(), 3), "-", round(net.res_bus.vm_pu.max(), 3))

converged: True
slack P (MW): 894.1
vm range: 0.943 - 1.04


In [70]:
# --- Block 6: ratings + violation checker ---
import numpy as np
import pandas as pd

buses_df  = pd.read_csv("buses.csv")
lines_df  = pd.read_csv("lines.csv")
trafos_df = pd.read_csv("trafos.csv")

# net only contains in-service lines/trafos, created in CSV order, so line the ratings up the same way
l_on = lines_df[lines_df["Line Status"] == 1].reset_index(drop=True)
t_on = trafos_df[trafos_df["Status"] == 1].reset_index(drop=True)
assert len(l_on) == len(net.line) and len(t_on) == len(net.trafo), "row counts don't match the network"

net.line["normal_mva"]  = l_on["Normal Rating"].values
net.line["emerg_mva"]   = l_on["Emergency Rating"].values
net.trafo["normal_mva"] = t_on["Normal Rating"].values
net.trafo["emerg_mva"]  = t_on["Emergency Rating"].values
net.bus["emerg_min"]    = buses_df["Emergency Min Voltage"].values
net.bus["emerg_max"]    = buses_df["Emergency Max Voltage"].values

assert len(buses_df) == len(net.bus), "bus count doesn't match"
net.bus["min_vm_pu"] = buses_df["Normal Min Voltage"].values
net.bus["max_vm_pu"] = buses_df["Normal Max Voltage"].values

def violations(net, emergency=False):
    """Return (voltage, line, trafo) violation tables for the last solved case."""
    vm = net.res_bus.vm_pu
    lo = net.bus.emerg_min if emergency else net.bus.min_vm_pu
    hi = net.bus.emerg_max if emergency else net.bus.max_vm_pu
    v = pd.DataFrame({"bus": net.bus.name, "vm_pu": vm, "min": lo, "max": hi})
    v = v[(v.vm_pu < v["min"]) | (v.vm_pu > v["max"])]          # NaN (islanded) buses fall out here

    rate = "emerg_mva" if emergency else "normal_mva"
    l_mva = np.maximum(np.hypot(net.res_line.p_from_mw, net.res_line.q_from_mvar),
                       np.hypot(net.res_line.p_to_mw,   net.res_line.q_to_mvar))
    l = pd.DataFrame({"line": net.line.name, "mva": l_mva, "rating": net.line[rate]})
    l["pct"] = 100 * l.mva / l.rating
    l = l[l.pct > 100]

    t_mva = np.maximum(np.hypot(net.res_trafo.p_hv_mw, net.res_trafo.q_hv_mvar),
                       np.hypot(net.res_trafo.p_lv_mw, net.res_trafo.q_lv_mvar))
    t = pd.DataFrame({"trafo": net.trafo.name, "mva": t_mva, "rating": net.trafo[rate]})
    t["pct"] = 100 * t.mva / t.rating
    t = t[t.pct > 100]
    return v, l, t


v, l, t = violations(net, emergency=False)
print("N-0 violations -> voltage:", len(v), " lines:", len(l), " trafos:", len(t))
print(v.round(3).to_string())

N-0 violations -> voltage: 9  lines: 0  trafos: 0
     bus  vm_pu   min   max
26    27  0.947  0.95  1.05
27    28  0.946  0.95  1.05
181  182  0.949  0.95  1.05
207  208  0.949  0.95  1.05
208  209  0.948  0.95  1.05
355  356  0.944  0.95  1.05
356  357  0.943  0.95  1.05
381  382  0.944  0.95  1.05
382  383  0.943  0.95  1.05


In [71]:
def run_n1(net):
    pp.runpp(net, numba=False, init="dc")                  # solve the case you were handed
    v0, _, _ = violations(net, emergency=False)
    base_buses = set(v0.bus)                                # buses already low before any outage

    def one_case(name):
        v_out, l_out, t_out = violations(net, emergency=True)
        return {
            "outage": name,
            "status": "ok",
            "islanded_buses": net.res_bus.vm_pu.isna().sum(),
            "n_voltage_viol": len(v_out),
            "n_new_voltage_viol": len(set(v_out.bus) - base_buses),
            "n_line_viol": len(l_out),
            "n_trafo_viol": len(t_out),
            "min_vm_pu": net.res_bus.vm_pu.min(),
            "worst_loading_pct": pd.concat([l_out.pct, t_out.pct]).max(),
        }

    rows, rows_t = [], []
    for i in net.line.index:
        name = net.line.at[i, "name"]
        net.line.at[i, "in_service"] = False
        try:
            pp.runpp(net, numba=False, init="dc")
            rows.append(one_case(name))
        except pp.LoadflowNotConverged:
            rows.append({"outage": name, "status": "did not converge"})
        finally:
            net.line.at[i, "in_service"] = True

    for i in net.trafo.index:
        name = net.trafo.at[i, "name"]
        net.trafo.at[i, "in_service"] = False
        try:
            pp.runpp(net, numba=False, init="dc")
            rows_t.append(one_case(name))
        except pp.LoadflowNotConverged:
            rows_t.append({"outage": name, "status": "did not converge"})
        finally:
            net.trafo.at[i, "in_service"] = True

    results, results_t = pd.DataFrame(rows), pd.DataFrame(rows_t)
    results["element"] = "line"
    results_t["element"] = "trafo"
    return pd.concat([results, results_t], ignore_index=True)

In [72]:
base_results = run_n1(net)
base_results.to_csv("n1_base_case.csv", index=False)
p = base_results[(base_results.n_voltage_viol > 0) | (base_results.n_line_viol > 0) |
                 (base_results.n_trafo_viol > 0) | (base_results.status != "ok")]
print(len(base_results), "outages;", len(p), "with problems")   # expect 597 and 29

597 outages; 29 with problems


In [73]:
import copy
net_dc = copy.deepcopy(net)
bus_idx = net_dc.bus.index[net_dc.bus.name == "242"]
print(bus_idx)   # should be exactly one index

Index([241], dtype='int64')


In [74]:
#pp.create_load(net_dc, bus=..., p_mw=..., q_mvar=..., name=...) adds it. Use bus_idx[0] for the bus.
#Q comes from your power factor: q_mvar = p_mw * tan(arccos(pf)).
#Use numpy for tan and arccos. 0.95 is a common choice. Whatever you pick, write it down for the README.
#Give the load a name like "data_center", so you can find it later in net_dc.load.

p_mw = 300
pf = 0.95
q_mvar = p_mw * np.tan(np.arccos(pf))

pp.create_load(net_dc, bus=bus_idx[0], p_mw=p_mw, q_mvar=q_mvar, name="data_center")
print(round(q_mvar, 1))      # should be about 98.6
print(net_dc.load.tail(1))   # confirm the load is there


98.6
            name  bus   p_mw     q_mvar  const_z_percent  const_i_percent  \
206  data_center  241  300.0  98.605232              0.0              0.0   

     sn_mva  scaling  in_service type  
206     NaN      1.0        True  wye  


In [77]:
pp.runpp(net_dc, numba=False, init="dc")
v, l, t = violations(net_dc, emergency=False)
print(len(v), len(l), len(t))
print(net_dc.res_bus.loc[bus_idx[0]])

16 2 0
vm_pu          0.882121
va_degree    -32.793182
p_mw         300.000000
q_mvar        98.605232
Name: 241, dtype: float64


In [76]:
pp.runpp(net_dc, numba=False, init="dc")
v, l, t = violations(net_dc, emergency=False)
print(len(v), len(l), len(t))
print(v.sort_values("vm_pu").to_string())
print(l)

          line         mva  rating         pct
91    87-141-1  320.976885  320.29  100.214457
282  276-320-1  343.032148  320.29  107.100486
     bus     vm_pu   min   max
242  243  0.880768  0.95  1.05
241  242  0.882121  0.95  1.05
320  321  0.904531  0.95  1.05
319  320  0.907194  0.95  1.05
356  357  0.940988  0.95  1.05
355  356  0.941858  0.95  1.05
382  383  0.942062  0.95  1.05
381  382  0.942538  0.95  1.05
208  209  0.945344  0.95  1.05
27    28  0.945567  0.95  1.05


In [78]:
for q in [0, 25, 50, 75, 100]:
    test = copy.deepcopy(net_dc)
    if q > 0:
        pp.create_shunt(test, bus=bus_idx[0], q_mvar=-q, name="cap_242")
    pp.runpp(test, numba=False, init="dc")
    v, l, t = violations(test, emergency=False)
    print(q, "MVAr -> V242:", round(test.res_bus.at[bus_idx[0], "vm_pu"], 3),
          "| voltage viol:", len(v), "| line viol:", len(l))

0 MVAr -> V242: 0.882 | voltage viol: 16 | line viol: 2
25 MVAr -> V242: 0.893 | voltage viol: 16 | line viol: 2
50 MVAr -> V242: 0.905 | voltage viol: 16 | line viol: 2
75 MVAr -> V242: 0.917 | voltage viol: 16 | line viol: 2
100 MVAr -> V242: 0.929 | voltage viol: 16 | line viol: 2


In [79]:
pp.create_shunt(net_dc, bus=bus_idx[0], q_mvar=-150, name="cap_242")

0

In [80]:
print(net_dc.shunt)

   bus     name  q_mvar  p_mw  vn_kv  step  max_step  in_service
0  241  cap_242  -150.0   0.0  138.0     1         1        True


In [81]:
pp.runpp(net_dc, numba=False, init="dc")
v, l, t = violations(net_dc, emergency=False)
vm = net_dc.res_bus.vm_pu
print(len(v), len(l), len(t))
print("V242:", round(vm[241], 3), "V243:", round(vm[242], 3),
      "V320:", round(vm[319], 3), "V321:", round(vm[320], 3))
print(l)

12 1 0
V242: 0.954 V243: 0.952 V320: 0.954 V321: 0.951
        line         mva  rating         pct
91  87-141-1  320.928129  320.29  100.199235


In [83]:
dc150 = run_n1(net_dc)
dc150.to_csv("n1_datacenter_150.csv", index=False)

In [84]:
net_dc.shunt.at[0, "q_mvar"] = -200
dc200 = run_n1(net_dc)
dc200.to_csv("n1_datacenter_200.csv", index=False)

In [85]:
for name in ["242-287-1", "242-320-1"]:
    test = copy.deepcopy(net_dc)
    i = test.line.index[test.line.name == name][0]
    test.line.at[i, "in_service"] = False
    pp.runpp(test, numba=False, init="dc")
    v, l, t = violations(test, emergency=True)
    print("outage:", name); print(l)

outage: 242-287-1
          line         mva   rating         pct
253  242-320-1  379.805533  352.319  107.801604
282  276-320-1  561.797862  352.319  159.457157
outage: 242-320-1
          line         mva   rating         pct
254  242-287-1  387.492289  352.319  109.983364
